# 00 - Tạo lớp "Khác" (chạy 1 lần)

Người phụ trách: Hân

Notebook này chỉ làm một việc: tạo thư mục ảnh cho lớp `other` ("Khác") và nén thành `other_split.zip` trên Drive.
Notebook **01b** sẽ giải nén file zip này để huấn luyện. Chỉ cần chạy **một lần**; chạy lại vẫn ra đúng bộ ảnh cũ
vì cách chọn ảnh dùng seed cố định (42).

Lớp Khác gồm 736 ảnh, lấy từ bản sao trên Hugging Face của các bộ dữ liệu công khai:
- **Hoa ngoài 5 loài** (368 ảnh): Oxford 102 Flowers, bỏ các loài trùng hoặc rất giống 5 loài của Anthea.
- **Không phải hoa** (368 ảnh): Caltech-101 và Imagenette.

Chia cố định: train 516 / val 110 / test 110. Không cần GPU.

Cell 1 – Thư viện, kết nối Google Drive, đường dẫn

In [ ]:
import os
import io
import math
import random
import shutil

import requests
import pandas as pd
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

ROOT      = '/content/drive/MyDrive/flower-classifier'
SPLIT_ZIP = f'{ROOT}/data/tf_flowers_split.zip'   # bản chia 5 loài hoa (notebook 01), chỉ dùng để đếm ảnh
OTHER_ZIP = f'{ROOT}/data/other_split.zip'        # KẾT QUẢ của notebook này
SPLIT_DIR = '/content/tf_flowers_split'
OTHER_DIR = '/content/other_split'

FLOWERS = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']

Cell 2 – Danh sách bỏ qua và địa chỉ tải dữ liệu

In [ ]:
# Hoa trùng hoặc rất giống 5 loài của Anthea: bỏ, để lớp Khác không chứa ảnh mà người xem cũng gọi là cúc, hồng…
OXFORD_SKIP = ['oxeye daisy', 'barbeton daisy', 'common dandelion', "colt's foot",
               'rose', 'sunflower', 'black-eyed susan']

# Caltech-101: bỏ 3 nhóm là hoa, và background_google (ảnh ngẫu nhiên, có thể lẫn hoa)
CALTECH_SKIP = ['lotus', 'sunflower', 'water_lilly', 'background_google']

HF_API  = 'https://huggingface.co/api/datasets'
HF_INFO = 'https://datasets-server.huggingface.co/info?dataset='

Cell 3 – Hàm tải một bộ ảnh và hàm lấy mẫu đều

In [ ]:
def load_by_class(dataset, skip):
    """Tải một bộ ảnh trên Hugging Face (dạng file parquet).
    Trả về dict: {tên lớp con: [ảnh 1 (bytes), ảnh 2 (bytes), ...]}, đã bỏ các lớp con nằm trong skip."""

    # Bước 1: lấy tên các lớp con. names[0] là tên của nhãn số 0, names[1] là tên của nhãn số 1, ...
    response = requests.get(HF_INFO + dataset)
    info = response.json()
    names = info['dataset_info']['default']['features']['label']['names']

    # Bước 2: lấy link các file parquet, dạng {'train': [link, ...], 'test': [link, ...]}
    response = requests.get(f'{HF_API}/{dataset}/parquet/default')
    files = response.json()

    # Bước 3: tải từng file, xếp từng ảnh vào đúng lớp con. Gộp mọi tập con (train, validation, test)
    by_class = {}
    for split_name in files:
        for link in files[split_name]:
            response = requests.get(link)
            response.raise_for_status()                             # tải lỗi thì dừng luôn, báo lỗi
            table = pd.read_parquet(io.BytesIO(response.content))   # bảng: mỗi dòng có cột image và cột label

            image_list = list(table['image'])    # mỗi phần tử là dict, ảnh nằm ở khóa 'bytes'
            label_list = list(table['label'])    # mỗi phần tử là 1 số nguyên (nhãn)
            for row in range(len(label_list)):
                class_name = names[label_list[row]].strip()
                if class_name in skip:
                    continue                                 # lớp con bị bỏ -> bỏ qua ảnh này
                if class_name not in by_class:
                    by_class[class_name] = []                # gặp lớp con mới -> tạo danh sách rỗng
                by_class[class_name].append(image_list[row]['bytes'])
    return by_class


def sample_evenly(by_class, n, rng):
    """Lấy n ảnh, chia đều cho các lớp con (mỗi lớp con vài tấm) để lớp Khác đa dạng nhất."""
    per_class = math.ceil(n / len(by_class))     # số ảnh mỗi lớp con, làm tròn lên
    picked = []
    for class_name in sorted(by_class):          # duyệt theo thứ tự ABC để lần nào chạy cũng giống nhau
        images = by_class[class_name]
        k = min(per_class, len(images))          # lớp con ít ảnh hơn per_class thì lấy hết
        chosen = rng.sample(images, k)           # chọn ngẫu nhiên k ảnh, không trùng
        for image in chosen:
            picked.append(image)
    rng.shuffle(picked)
    return picked[:n]                            # làm tròn lên nên có thể dư vài ảnh -> cắt còn đúng n

Cell 4 – Tính số ảnh của lớp Khác (bằng cỡ trung bình một loài hoa)

In [ ]:
shutil.rmtree(SPLIT_DIR, ignore_errors=True)
shutil.unpack_archive(SPLIT_ZIP, SPLIT_DIR)

# Đếm ảnh train của từng loài hoa
total_train = 0
for flower in FLOWERS:
    num = len(os.listdir(f'{SPLIT_DIR}/train/{flower}'))
    print(flower, num)
    total_train = total_train + num

# Số ảnh train trung bình / 0.70 = tổng cả 3 tập (vì train chiếm 70%)
average = total_train / len(FLOWERS)
total = round(average / 0.70)
half = total // 2                       # nửa hoa ngoài 5 loài, nửa không phải hoa
print(f'Trung bình {average} ảnh train mỗi loài -> lớp Khác: {total} ảnh ({half} + {total - half})')

Cell 5 – Tải 3 bộ ảnh (khoảng 800 MB, vài phút)

In [ ]:
oxford     = load_by_class('dpdl-benchmark/oxford_flowers102', OXFORD_SKIP)   # Oxford 102 Flowers
caltech    = load_by_class('flwrlabs/caltech101', CALTECH_SKIP)              # Caltech-101
imagenette = load_by_class('johnowhitaker/imagenette2-320', [])              # Imagenette 320px

# Gộp Caltech và Imagenette thành nhóm "không phải hoa"
not_flower = {}
for name in caltech:
    not_flower[name] = caltech[name]
for name in imagenette:
    not_flower[name] = imagenette[name]

print(f'Hoa ngoài 5 loài: {len(oxford)} loài | Không phải hoa: {len(not_flower)} nhóm')

Cell 6 – Chọn ảnh và chia 3 tập

In [ ]:
rng = random.Random(42)   # seed riêng cố định -> lần nào chạy cũng chọn đúng các ảnh này

# Mỗi phần tử là 1 cặp (loại, ảnh). Loại sẽ nằm trong tên file ("hoakhac" / "khonghoa") để 01b đo riêng từng nửa.
# Giữ đúng thứ tự: lấy Oxford trước rồi mới lấy nhóm không phải hoa, để rng chọn ra đúng các ảnh như cũ
images = []
for data in sample_evenly(oxford, half, rng):
    images.append(('hoakhac', data))
for data in sample_evenly(not_flower, total - half, rng):
    images.append(('khonghoa', data))
rng.shuffle(images)

# Chia 15% test, 15% val, còn lại (70%) là train
n = int(len(images) * 0.15)
test_images  = images[:n]
val_images   = images[n:2 * n]
train_images = images[2 * n:]

parts = {'test': test_images, 'val': val_images, 'train': train_images}
for split in parts:
    print(split, len(parts[split]), 'ảnh')

Cell 7 – Lưu ảnh ra thư mục và nén lên Drive

In [ ]:
shutil.rmtree(OTHER_DIR, ignore_errors=True)    # bắt đầu từ thư mục trống

for split in parts:
    folder = f'{OTHER_DIR}/{split}/other'
    os.makedirs(folder, exist_ok=True)
    items = parts[split]
    for i in range(len(items)):
        kind = items[i][0]                             # 'hoakhac' hoặc 'khonghoa'
        data = items[i][1]                             # nội dung ảnh (bytes)
        number = str(i).zfill(4)                       # 7 -> '0007'
        path = f'{folder}/{kind}_{split}_{number}.jpg'
        # Lưu lại thành JPEG màu RGB: Caltech có ảnh xám, Imagenette có vài ảnh CMYK mà TensorFlow không đọc được
        img = Image.open(io.BytesIO(data))
        img = img.convert('RGB')
        img.save(path, quality=95)

# Nén lên Drive. make_archive tự thêm đuôi .zip nên phải bỏ đuôi .zip khỏi tên trước
zip_name = OTHER_ZIP.replace('.zip', '')
shutil.make_archive(zip_name, 'zip', OTHER_DIR)
print('Đã lưu:', OTHER_ZIP)

Cell 8 – Kiểm tra lại số ảnh từng nửa

In [ ]:
for split in ['train', 'val', 'test']:
    file_names = os.listdir(f'{OTHER_DIR}/{split}/other')
    num_hoakhac = 0
    num_khonghoa = 0
    for file_name in file_names:
        if file_name.startswith('hoakhac_'):
            num_hoakhac = num_hoakhac + 1
        else:
            num_khonghoa = num_khonghoa + 1
    print(f'{split.ljust(5)}: {len(file_names)} ảnh = {num_hoakhac} hoa ngoài 5 loài + {num_khonghoa} không phải hoa')

size_mb = os.path.getsize(OTHER_ZIP) / 1e6
print(f'Dung lượng other_split.zip: {size_mb:.1f} MB')